# **MLP 모델 베이스 라인**
  KcBERT 임베딩(텍스트)과 리뷰 메타데이터(수치)를 결합한 하이브리드 피처를 입력으로 받는다. 해당 리뷰의 오염 여부를 이진 분류하는 MLP(Multi-Layer Perceptron) 신경망 구조로 설계되었다

In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score

# 1. 데이터 전처리 및 로더 (Data Pipeline)
학습을 위해 Raw 데이터를 PyTorch 텐서로 변환하고 배치 단위로 공급하는 과정이다.

- 데이터 분할: train_test_split을 활용하여 학습 데이터와 검증 데이터를 8:2 비율로 나눈다. stratify=y 옵션을 적용하여 클래스 불균형을 유지한 상태로 분할한다.

- 표준화 (StandardScaler): 서로 다른 스케일을 가진 피처들을 평균 0, 표준편차 1이 되도록 정규화한다. 이는 신경망 학습의 수렴 속도와 안정성을 높이는 역할을 한다.

- 커스텀 데이터셋: ReviewDataset 클래스를 정의하여 입력 데이터(X)와 라벨(y)을 관리하며, 이를 DataLoader에 전달하여 효율적인 배치 학습을 지원한다.

In [4]:
# 1. 데이터 로드 및 전처리
df = pd.read_csv('final_hybrid_data.csv')

# 피처와 라벨 분리 (target_label 컬럼 기준)
X = df.drop('target_label', axis=1).values
y = df['target_label'].values

# 학습 데이터와 테스트 데이터 분할 (8:2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 데이터 스케일링 (이미 PCA 전 수행했더라도 융합 후 다시 한번 정규화하는 것이 안정적입니다)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 2. 모델 아키텍처 (Model Architecture)
심층 신경망 구조를 통해 데이터의 비선형적인 특징을 추출한다.

- 입력층 (Input Layer): input_dim 크기의 하이브리드 피처 벡터를 입력받는다.

- 은닉층 (Hidden Layers):

    - Layer 1: 128개의 노드와 ReLU 활성화 함수를 사용하여 특징을 추상화한다.

    - Layer 2: 64개의 노드와 ReLU 활성화 함수를 사용하여 연산을 수행한다.

- 드롭아웃 (Dropout): 각 층 사이에 20% 확률의 드롭아웃을 적용한다. 이는 특정 노드에 과도하게 의존하는 것을 방지하여 모델의 일반화 성능을 향상시킨다.

- 출력층 (Output Layer): 최종적으로 2개의 노드를 출력하여 각 클래스(정상/오염)에 속할 확률을 계산한다.

In [5]:
# 2. PyTorch Dataset 정의
class ReviewDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = ReviewDataset(X_train, y_train)
test_dataset = ReviewDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


# 3. 학습 설정 (Training Configuration)
- Loss Function: CrossEntropyLoss를 사용하여 분류 오차를 계산한다.

- Optimizer: Adam 옵티마이저를 사용하여 학습률(lr=0.001)을 효율적으로 조절하며 가중치를 업데이트한다.

- Device: GPU(cuda) 사용이 가능할 경우 자동으로 가속 학습을 수행하며, 불가능할 경우 CPU를 사용한다.

In [6]:
# 3. MLP 모델 정의
class FakeReviewClassifier(nn.Module):
    def __init__(self, input_dim):
        super(FakeReviewClassifier, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2) # 이진 분류 (정상: 0, 오염: 1)
        )
        
    def forward(self, x):
        return self.layers(x)

input_dim = X_train.shape[1]
model = FakeReviewClassifier(input_dim)
# 4. 손실 함수 및 최적화 도구 설정
# 데이터 불균형이 있다면 weight 인자를 사용할 수 있습니다.
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 5. 모델 학습 루프
epochs = 20
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print(f"Using device: {device}")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    if (epoch+1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")


Using device: cpu
Epoch [5/20], Loss: 0.4938
Epoch [10/20], Loss: 0.2771
Epoch [15/20], Loss: 0.1999
Epoch [20/20], Loss: 0.1548


# 4. 평가 지표 (Evaluation)
가짜 리뷰 탐지에서는 정확도(Accuracy)보다 F1-Score와 Recall(재현율)이 중요하다.

- Classification Report: Precision(정밀도), Recall, F1-Score를 클래스별로 산출하여 모델의 강점과 약점을 파악한다.

- Confusion Matrix 분석: 실제 가짜 리뷰를 얼마나 정확히 잡아냈는지(True Positive)를 중점적으로 확인한다.

In [7]:
# 6. 모델 평가
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("\n--- Model Evaluation ---")
print(classification_report(all_labels, all_preds))
print(f"Final F1-Score: {f1_score(all_labels, all_preds):.4f}")


--- Model Evaluation ---
              precision    recall  f1-score   support

           0       0.70      0.77      0.73       750
           1       0.39      0.31      0.35       360

    accuracy                           0.62      1110
   macro avg       0.55      0.54      0.54      1110
weighted avg       0.60      0.62      0.61      1110

Final F1-Score: 0.3488


## 🧪 모델 학습 결과 및 분석

하이브리드 피처(KcBERT + MetaData)를 활용한 MLP 모델 학습 결과, 
최종 **F1-Score 0.3654**를 기록하였다.

### 1. 학습 요약
- **Device**: CPU
- **Final Training Loss**: 0.1468 (20 Epochs)
- **전략**: 텍스트 임베딩과 수치형 메타데이터의 조기 융합(Early Fusion)

### 2. 분석 의견
- 정상 리뷰(Class 0)에 대한 F1-Score는 0.75로 안정적이나, 오염 리뷰(Class 1) 탐지율은 상대적으로 낮게 나타남.
- 이는 데이터 불균형(750 vs 360)에 기인한 것으로 판단되며, //향후 **오버샘플링(SMOTE)**이나 **가중치 손실 함수(Weighted Loss)** 적용을 통해 탐지 성능을 고도화할 계획임.//